In [ ]:
pip install transformers datasets torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from transformers import TrainingArguments, Trainer, AutoTokenizer
from transformers import AutoModelForSeq2SeqLM
from peft import LoraConfig, TaskType
from datasets import Dataset
import torch
import pandas as pd

In [ ]:
df = pd.read_csv('bank-full.csv', sep=';')
df.head(4)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
8084,35,services,married,secondary,no,152,yes,no,unknown,2,jun,563,1,-1,0,unknown,yes
29020,42,blue-collar,married,primary,no,199,yes,no,cellular,2,feb,272,1,-1,0,unknown,no
9257,39,admin.,married,secondary,no,1595,yes,no,unknown,5,jun,102,2,-1,0,unknown,no
31090,21,blue-collar,single,secondary,no,2265,no,no,cellular,17,feb,92,2,-1,0,unknown,yes
41357,55,management,married,unknown,no,1504,yes,no,cellular,31,aug,186,3,101,2,success,yes


In [ ]:
df['y'].tail(10)

,y
45201,yes
45202,yes
45203,yes
45204,yes
45205,yes
45206,yes
45207,yes
45208,yes
45209,no
45210,no


In [ ]:

# Преобразование данных в текст
def concatenate_text_bank(x):
    # Преобразуем бинарные значения в текст
    dft = 'has a credit in default' if x['default'] == 'yes' else 'has no credit in default'
    hsg = 'has housing loan' if x['housing'] == 'yes' else "doesn't have housing loan"
    ln = 'has a personal loan' if x['loan'] == 'yes' else 'has no personal loan'

    # Формируем текст описания
    full_text = (
        f"This customer is {x['age']} years old, "
        f"with {x['job']} occupation, "
        f"is {x['marital']}, "
        f"with {x['education']} level of education, "
        f"{dft}, "
        f"with average yearly balance of {x['balance']} euros, "
        f"{hsg}, "
        f"{ln}, "
        f"contacted via {x['contact']} type of contact, "
        f"on {x['day']} day of {x['month']}, "
        f"with the last call duration of {x['duration']} seconds, "
        f"{x['campaign']} times of call in current marketing campaign, "
        f"with {x['pdays']} days of pass between contacts, "
        f"{x['previous']} times of contacts in the previous campaign, "
        f"and last poutcome is {x['poutcome']}."
    )
    return full_text

In [ ]:
df['text'] = df.apply(lambda x: concatenate_text_bank(x), axis=1)
df['text'].iloc[3]

'This customer is 47 years old, with blue-collar occupation, is married, with unknown level of education, has no credit in default, with average yearly balance of 1506 euros, has housing loan, has no personal loan, contacted via unknown type of contact, on 5 day of may, with the last call duration of 92 seconds, 1 times of call in current marketing campaign, with -1 days of pass between contacts, 0 times of contacts in the previous campaign, and last poutcome is unknown.'

In [ ]:
# Преобразование в формат Dataset
dataset = Dataset.from_pandas(df[['text', 'y']])
dataset

Dataset({
    features: ['text', 'y'],
    num_rows: 45211
})

In [ ]:
# Загрузка модели и токенизатора
model = AutoModelForSeq2SeqLM.from_pretrained("bigScience/T0_3B")
tokenizer = AutoTokenizer.from_pretrained("bigScience/T0_3B")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/11.4G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.86k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [ ]:
# Перемещение модели на GPU
model.to(device)

PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): T5ForConditionalGeneration(
      (shared): Embedding(32128, 2048)
      (encoder): T5Stack(
        (embed_tokens): Embedding(32128, 2048)
        (block): ModuleList(
          (0): T5Block(
            (layer): ModuleList(
              (0): T5LayerSelfAttention(
                (SelfAttention): T5Attention(
                  (q): lora.Linear(
                    (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=2048, out_features=8, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=8, out_features=2048, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
         

In [ ]:
def tokenize_function(examples):
    inputs = tokenizer(examples['text'], padding="max_length", truncation=True, max_length=64, return_tensors="pt")
    labels = tokenizer(examples['y'], padding="max_length", truncation=True, max_length=64, return_tensors="pt")
    inputs['labels'] = labels['input_ids']
    return inputs

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/45211 [00:00<?, ? examples/s]

In [ ]:
tokenized_dataset

Dataset({
    features: ['text', 'y', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 45211
})

In [ ]:
# Разделение данных на train и test
split_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42)

train_data = split_dataset['train']
test_data = split_dataset['test']

In [ ]:
from collections import Counter

y_values = train_data['y']
if isinstance(y_values, list):
    print(Counter(y_values))

Counter({'no': 31966, 'yes': 4202})


In [ ]:
y_values_test = test_data['y']
if isinstance(y_values_test, list):
    print(Counter(y_values_test))

Counter({'no': 7956, 'yes': 1087})


In [ ]:
# Удаляем ненужные колонки
train_data = train_data.remove_columns(['text', 'y'])
test_data = test_data.remove_columns(['text', 'y'])

In [ ]:
train_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

In [ ]:
# Проверка формы input_ids и attention_mask
print("Форма input_ids:", train_data[0]['input_ids'].shape)
print("Форма attention_mask:", train_data[0]['attention_mask'].shape)
print("Форма labels:", train_data[0]['labels'].shape)


Форма input_ids: torch.Size([64])
Форма attention_mask: torch.Size([64])
Форма labels: torch.Size([64])


In [ ]:
train_data['attention_mask'][0]

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

In [ ]:
train_data['input_ids'][0]

tensor([  100,   884,    19,  3538,   203,   625,     6,    28, 10473,     5,
        13792,     6,    19,   712,     6,    28,  6980,   593,    13,  1073,
            6,    65,   150,   998,    16,  4647,     6,    28,  1348,     3,
        22286,  2109,    13,     3,    18,  2668, 10186,     6,   744,    31,
           17,    43,  3499,  2289,     6,    65,   150,   525,  2289,     6,
            3, 12655,  1009,     3, 10791,   686,    13,   574,     6,    30,
         2838,   239,    13,     1])

In [ ]:
train_data['labels'][0]

tensor([150,   1,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0])

In [ ]:
# Заморозка всех слоев в энкодере
for param in model.encoder.parameters():
    param.requires_grad = False

# Заморозка всех слоев в декодере, кроме 23-го блока
for i, layer in enumerate(model.decoder.block):
    if i != 23:  # Указание на 23-й слой (индекс 23)
        for param in layer.parameters():
            param.requires_grad = False

# Проверка заморозки
for name, param in model.named_parameters():
    print(f"{name}: requires_grad = {param.requires_grad}")

base_model.model.shared.weight: requires_grad = False
base_model.model.encoder.block.0.layer.0.SelfAttention.q.base_layer.weight: requires_grad = False
base_model.model.encoder.block.0.layer.0.SelfAttention.q.lora_A.default.weight: requires_grad = False
base_model.model.encoder.block.0.layer.0.SelfAttention.q.lora_B.default.weight: requires_grad = False
base_model.model.encoder.block.0.layer.0.SelfAttention.k.weight: requires_grad = False
base_model.model.encoder.block.0.layer.0.SelfAttention.v.base_layer.weight: requires_grad = False
base_model.model.encoder.block.0.layer.0.SelfAttention.v.lora_A.default.weight: requires_grad = False
base_model.model.encoder.block.0.layer.0.SelfAttention.v.lora_B.default.weight: requires_grad = False
base_model.model.encoder.block.0.layer.0.SelfAttention.o.weight: requires_grad = False
base_model.model.encoder.block.0.layer.0.SelfAttention.relative_attention_bias.weight: requires_grad = False
base_model.model.encoder.block.0.layer.0.layer_norm.weight:

In [ ]:
# torch.cuda.empty_cache()

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    run_name="my_t0_experiment",
    report_to="none",
    eval_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
)

In [ ]:
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,0.011600,0.006120


In [ ]:
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score

In [ ]:
# Функция для декодирования меток из тензоров в текст
def decode_labels(labels):
    decoded_labels = []
    for label in labels:
        decoded_label = tokenizer.decode(label, skip_special_tokens=True).strip()
        decoded_labels.append(decoded_label)
    return decoded_labels

In [ ]:
def zero_shot_predict_tokens(input_ids):
    # Создаем промпт для модели
    prompt = "Does this client subscribe to a term deposit? Yes or no?"

    # Токенизируем промпт
    prompt_ids = tokenizer(prompt, return_tensors="pt", max_length=64, truncation=True).to("cuda")

    # Объединяем input_ids и prompt_ids
    input_ids = input_ids.to("cuda")
    combined_input_ids = torch.cat([input_ids, prompt_ids['input_ids']], dim=-1)

    # Генерируем ответ модели
    with torch.no_grad():
        output_ids = model.generate(input_ids=combined_input_ids, max_length=64)  # Исправлено здесь

    # Декодируем ответ модели в текст
    predicted_class_text = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    return 'yes' if predicted_class_text.lower() == 'yes' else 'no', output_ids

In [ ]:
# Функция для получения предсказаний на тестовых данных
def get_predictions(test_data):
    predictions = []
    true_labels = []

    # Декодируем истинные метки
    true_labels_text = decode_labels(test_data['labels'])

    for i, example in enumerate(test_data):
        # Получаем предсказание
        predicted_label = zero_shot_predict_tokens(example['input_ids'].unsqueeze(0))

        # Преобразуем метки в числовой формат
        true_labels.append(1 if true_labels_text[i] == 'yes' else 0)
        predictions.append(1 if predicted_label == 'yes' else 0)

    return true_labels, predictions

In [ ]:
# Получаем предсказания и истинные метки
true_labels, predictions = get_predictions(test_data)

In [ ]:
# Вычисляем метрики
roc_auc = roc_auc_score(true_labels, predictions)
f1 = f1_score(true_labels, predictions)
accuracy = accuracy_score(true_labels, predictions)
precision = precision_score(true_labels, predictions)
recall = recall_score(true_labels, predictions)

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Выводим результаты
print(f"ROC AUC: {roc_auc}")
print(f"F1 Score: {f1}")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")

ROC AUC: 0.5
F1 Score: 0.0
Accuracy: 0.8797965277009842
Precision: 0.0
Recall: 0.0


In [ ]:
from collections import Counter

prediction_counts = Counter(predictions)

print(prediction_counts)

Counter({0: 9043})
